# 03 — Eval

Phase 3 (Per-Field-Accuracy Baseline), Phase 4 (Iterations-Auswertung + Synthese-Tabelle), Phase 6 (Skalierung 7B + 3B-Halluzinations-Klassen).

Predictions kommen aus `02_extract.ipynb`. Gold ist eure `annotation/meine_gold.csv` aus Phase 2.

## Run-Header

| Feld | Wert |
|---|---|
| Predictions | `daten/predictions.jsonl` (12 Anzeigen, generiert von `02_extract.ipynb`) |
| Gold | `annotation/meine_gold.csv` (Hand-Annotation aus Phase 2) |
| Modell | `Qwen/Qwen2.5-7B-Instruct` (Baseline, ohne Iteration) |
| Datum | 2026-05-18 |


## Phase 3 — Baseline-Accuracy auf 12 Hand-Gold-Anzeigen

Hypothese-Cell *vor* der Eval: welches Feld haltet ihr für am stärksten / am schwächsten — und warum?

## Hypothese vor der Eval

Vor Berechnung der konkreten Accuracy-Zahlen erwarte ich folgende Verteilung 
über die 6 Schema-Felder, basierend auf:
(a) der manuellen Inspektion einzelner Predictions in `02_extract.ipynb`,  
(b) der Korpus-Struktur (35 Festanstellung + 20 Ausbildung, viele HO-stumme Anzeigen),  
(c) der Allianz-Anzeige mit leeren Pflichtfeldern.

### Erwartung nach Feld

| Feld | Erwartete Stärke | Begründung |
|---|---|---|
| `vertragsart` | **stärkstes Feld** | Die Unterscheidung Ausbildung vs. Festanstellung ist im Text textuell sehr eindeutig (Wort "Ausbildung" steht explizit). Im κ aus Phase 2 hatten wir hier 100 % zwischen den menschlichen Annotator:innen — das Modell sollte dem nicht weit nachstehen. |
| `homeoffice` | stark | Im Korpus dominieren "stumme" Anzeigen → triviales `nicht_genannt`. Die wenigen `teilweise`/`ja`-Fälle haben sehr klare Signalwörter. Risiko: Modell überinterpretiert "flexible Arbeitszeit" als Homeoffice. |
| `gehalt_zeitraum` | stark | Wenn `gehalt_min_eur` null ist, muss auch der Zeitraum null sein — Konsistenz-Regel ist einfach. Risiko: nur die Wilken-Ausbildung hat eine echte Zahl. |
| `skills_top3` | mittel | Set-Match toleriert Reihenfolge, aber das Modell halluziniert manchmal nicht-technische Begriffe (z.B. "agile", "frameworks" — bereits in den Predictions gesichtet). |
| `erfahrungslevel` | **schwächstes Feld** | Schon in Phase 2 hatten zwei Menschen κ = 0.625 (substanziell, aber nicht super). Die Schema-Definition ist subjektiv ("mehrjährig", "routiniert" → mid oder senior?). Das Modell hat in Zelle-6-Test "mid" gewählt, wo Gold "nicht_genannt" war. |
| `gehalt_min_eur` | schwach | Das Modell hat bereits bei der Wilken-Ausbildung "Ausbildungsjahr: 953,00 €" übersehen (siehe `02_extract.ipynb` Befunde). Vermutlich Modell-Problem: Tausender-Format/Komma irritiert es. |

**Zusammengefasst — meine Prognose:**
- **Top 2 (höchste Accuracy):** `vertragsart`, `homeoffice`
- **Bottom 2 (niedrigste Accuracy):** `erfahrungslevel`, `gehalt_min_eur`

Nach Berechnung der echten Zahlen unten reflektiere ich, wo ich richtig und 
wo ich falsch lag.

In [2]:
"""
Imports + Daten einlesen.
"""
import json
import csv
from pathlib import Path
from collections import Counter

# Predictions laden
predictions = {}
with open("../daten/predictions.jsonl", encoding="utf-8") as f:
    for line in f:
        p = json.loads(line)
        predictions[p["refnr"]] = p

# Gold laden  
gold = {}
with open("../annotation/meine_gold.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if not row["id"].strip():
            continue
        gold[row["id"]] = row

# Gemeinsame IDs
gemeinsame_ids = sorted(set(predictions) & set(gold))

print(f"Predictions: {len(predictions)} Einträge")
print(f"Gold:        {len(gold)} Einträge")
print(f"Gemeinsam:   {len(gemeinsame_ids)} Einträge (für Eval verwendet)")

# Sanity: falls nicht 12 → was fehlt?
if len(gemeinsame_ids) != 12:
    nur_pred = set(predictions) - set(gold)
    nur_gold = set(gold) - set(predictions)
    if nur_pred: print(f"⚠ Nur in Predictions: {nur_pred}")
    if nur_gold: print(f"⚠ Nur in Gold: {nur_gold}")

Predictions: 12 Einträge
Gold:        12 Einträge
Gemeinsam:   12 Einträge (für Eval verwendet)


In [3]:
"""
Per-Field-Accuracy berechnen.

Konventionen:
- homeoffice, vertragsart, erfahrungslevel, gehalt_zeitraum: exakter String-Match
- gehalt_min_eur: exakter Match auf Integer (oder beide null/leer)
- skills_top3: SET-MATCH (Reihenfolge egal, Strings lowercase + gestripped)
- Leere/None-Werte in Predictions → Feld zählt als FALSCH
"""

# Welche Felder evaluieren wir?
KATEGORIAL = ["homeoffice", "vertragsart", "erfahrungslevel", "gehalt_zeitraum"]


def normalisiere_gold_wert(wert: str):
    """CSV-Gold: leerer String -> None, sonst gestrippt."""
    if wert is None:
        return None
    wert = wert.strip()
    return wert if wert else None


def gehalt_match(pred, gold_str):
    """gehalt_min_eur: int oder None."""
    gold_norm = normalisiere_gold_wert(gold_str)
    gold_int = int(gold_norm) if gold_norm and gold_norm.isdigit() else None
    
    if pred is None and gold_int is None:
        return True
    if pred is None or gold_int is None:
        return False
    return int(pred) == gold_int


def skills_match(pred_liste, gold_str):
    """Set-Match auf Skills (lowercase, gestripped)."""
    # Gold ist Pipe-getrennter String aus CSV: "python|sql|r"
    gold_norm = normalisiere_gold_wert(gold_str)
    if gold_norm:
        gold_set = {s.strip().lower() for s in gold_norm.split("|") if s.strip()}
    else:
        gold_set = set()
    
    # Predictions: Liste aus JSON
    if pred_liste is None:
        pred_set = set()
    elif isinstance(pred_liste, list):
        pred_set = {str(s).strip().lower() for s in pred_liste if s}
    else:
        pred_set = set()
    
    return pred_set == gold_set


# Pro Feld zählen
ergebnisse = {feld: {"korrekt": 0, "total": 0, "fehler": []} for feld in 
              KATEGORIAL + ["gehalt_min_eur", "skills_top3"]}

for refnr in gemeinsame_ids:
    p = predictions[refnr]
    g = gold[refnr]
    
    # Kategoriale Felder
    for feld in KATEGORIAL:
        pred_wert = p.get(feld)
        # Predictions: None oder leerer String -> als "leer" werten
        if pred_wert is None or pred_wert == "":
            pred_norm = None
        else:
            pred_norm = str(pred_wert).strip()
        
        gold_norm = normalisiere_gold_wert(g.get(feld, ""))
        
        ergebnisse[feld]["total"] += 1
        if pred_norm == gold_norm:
            ergebnisse[feld]["korrekt"] += 1
        else:
            ergebnisse[feld]["fehler"].append({
                "refnr": refnr, "pred": pred_norm, "gold": gold_norm
            })
    
    # gehalt_min_eur
    pred_geh = p.get("gehalt_min_eur")
    ergebnisse["gehalt_min_eur"]["total"] += 1
    if gehalt_match(pred_geh, g.get("gehalt_min_eur", "")):
        ergebnisse["gehalt_min_eur"]["korrekt"] += 1
    else:
        ergebnisse["gehalt_min_eur"]["fehler"].append({
            "refnr": refnr, "pred": pred_geh, "gold": g.get("gehalt_min_eur", "")
        })
    
    # skills_top3
    pred_sk = p.get("skills_top3")
    ergebnisse["skills_top3"]["total"] += 1
    if skills_match(pred_sk, g.get("skills_top3", "")):
        ergebnisse["skills_top3"]["korrekt"] += 1
    else:
        ergebnisse["skills_top3"]["fehler"].append({
            "refnr": refnr, "pred": pred_sk, "gold": g.get("skills_top3", "")
        })


# Tabelle ausgeben
print(f"{'Feld':<20} {'Accuracy':>10} {'n_korrekt':>12} {'n_total':>10}")
print("─" * 56)
for feld in ["homeoffice", "vertragsart", "erfahrungslevel", 
             "gehalt_min_eur", "gehalt_zeitraum", "skills_top3"]:
    e = ergebnisse[feld]
    acc = e["korrekt"] / e["total"] if e["total"] else 0
    print(f"{feld:<20} {acc:>9.1%} {e['korrekt']:>12} {e['total']:>10}")

# Gesamtaccuracy
total_korrekt = sum(e["korrekt"] for e in ergebnisse.values())
total_zellen = sum(e["total"] for e in ergebnisse.values())
print("─" * 56)
print(f"{'GESAMT':<20} {total_korrekt/total_zellen:>9.1%} {total_korrekt:>12} {total_zellen:>10}")

# Hinweis auf leere Predictions
parse_fails = sum(
    1 for refnr in gemeinsame_ids 
    if predictions[refnr].get("parse_fail") 
    or not predictions[refnr].get("homeoffice")
)
print(f"\nHinweis: {parse_fails} Anzeige(n) mit fehlenden Pflichtfeldern "
      f"(zählen über alle Felder als Fail)")

Feld                   Accuracy    n_korrekt    n_total
────────────────────────────────────────────────────────
homeoffice               75.0%            9         12
vertragsart              91.7%           11         12
erfahrungslevel          66.7%            8         12
gehalt_min_eur           91.7%           11         12
gehalt_zeitraum          91.7%           11         12
skills_top3               8.3%            1         12
────────────────────────────────────────────────────────
GESAMT                   70.8%           51         72

Hinweis: 1 Anzeige(n) mit fehlenden Pflichtfeldern (zählen über alle Felder als Fail)


In [4]:
"""
Konkrete Fehlerfälle anschauen — die zwei schwächsten Felder.
Pflicht laut Aufgabenblatt: Hypothesen an Anzeigen begründen.
"""

print("="*80)
print("SCHWÄCHSTES FELD 1: skills_top3 (8.3% Accuracy)")
print("="*80)
for fall in ergebnisse["skills_top3"]["fehler"]:
    refnr = fall["refnr"]
    print(f"\nrefnr: {refnr}")
    print(f"  Pred:  {fall['pred']}")
    print(f"  Gold:  {fall['gold']}")

print("\n")
print("="*80)
print("SCHWÄCHSTES FELD 2: erfahrungslevel (66.7% Accuracy)")
print("="*80)
for fall in ergebnisse["erfahrungslevel"]["fehler"]:
    refnr = fall["refnr"]
    print(f"\nrefnr: {refnr}")
    print(f"  Pred:  {fall['pred']!r}")
    print(f"  Gold:  {fall['gold']!r}")

SCHWÄCHSTES FELD 1: skills_top3 (8.3% Accuracy)

refnr: 10000-1185445644-S
  Pred:  ['python', 'r', 'sql']
  Gold:  python|spark|ci/cd

refnr: 10001-1002026714-S
  Pred:  ['java', 'c#', 'sql']
  Gold:  

refnr: 10001-1002616378-S
  Pred:  ['python', 'pandas', 'scikit-learn']
  Gold:  python|machine learning|azure

refnr: 10001-1002678126-S
  Pred:  ['python', 'machine-learning', 'scikit-learn']
  Gold:  python|machine learning|git

refnr: 10001-1003030981-S
  Pred:  ['java', 'frameworks', 'testen']
  Gold:  

refnr: 13319-868490/1_607417LS-S
  Pred:  ['python', 'machine-learning', 'sql']
  Gold:  python|machine learning|sql

refnr: 13644-297872-S
  Pred:  ['python', 'pandas', 'scikit-learn']
  Gold:  python|machine learning|pandas

refnr: 15719-44156922-55-S
  Pred:  None
  Gold:  python|machine learning|SQL

refnr: 16818-100787927-S
  Pred:  ['java', 'delphi', 'sql']
  Gold:  

refnr: 16818-100814418-S
  Pred:  ['java', 'ms-office', 'agile']
  Gold:  java|ms office

refnr: 19384-93844

In [5]:
"""
Refactor: Eval-Logik als Funktion, damit wir sie für Baseline + Iteration A + Iteration B
wiederverwenden können, ohne Copy-Paste.

Liefert ein dict mit Accuracy pro Feld + Liste der Fehler (für Detail-Inspektion).
"""

def eval_predictions(predictions_pfad: str, gold_dict: dict) -> dict:
    """
    Lädt predictions.jsonl, joint mit gold, berechnet Per-Field-Accuracy.
    
    Returns:
        {
            "feld_name": {"korrekt": int, "total": int, "accuracy": float, "fehler": [...]}
            für alle 6 Felder + "_meta": {gemeinsame_ids, leere_predictions_count}
        }
    """
    preds = {}
    with open(predictions_pfad, encoding="utf-8") as f:
        for line in f:
            p = json.loads(line)
            preds[p["refnr"]] = p
    
    gem = sorted(set(preds) & set(gold_dict))
    KATEGORIAL = ["homeoffice", "vertragsart", "erfahrungslevel", "gehalt_zeitraum"]
    erg = {feld: {"korrekt": 0, "total": 0, "fehler": []} for feld in 
           KATEGORIAL + ["gehalt_min_eur", "skills_top3"]}
    leere_count = 0
    
    for refnr in gem:
        p = preds[refnr]
        g = gold_dict[refnr]
        
        # Leere Predictions zählen (Allianz-Fall)
        if not p.get("homeoffice"):
            leere_count += 1
        
        # Kategoriale Felder
        for feld in KATEGORIAL:
            pred_wert = p.get(feld)
            pred_norm = None if (pred_wert is None or pred_wert == "") else str(pred_wert).strip()
            gold_norm = normalisiere_gold_wert(g.get(feld, ""))
            erg[feld]["total"] += 1
            if pred_norm == gold_norm:
                erg[feld]["korrekt"] += 1
            else:
                erg[feld]["fehler"].append({"refnr": refnr, "pred": pred_norm, "gold": gold_norm})
        
        # gehalt_min_eur
        erg["gehalt_min_eur"]["total"] += 1
        if gehalt_match(p.get("gehalt_min_eur"), g.get("gehalt_min_eur", "")):
            erg["gehalt_min_eur"]["korrekt"] += 1
        else:
            erg["gehalt_min_eur"]["fehler"].append({
                "refnr": refnr, "pred": p.get("gehalt_min_eur"), "gold": g.get("gehalt_min_eur", "")
            })
        
        # skills_top3
        erg["skills_top3"]["total"] += 1
        if skills_match(p.get("skills_top3"), g.get("skills_top3", "")):
            erg["skills_top3"]["korrekt"] += 1
        else:
            erg["skills_top3"]["fehler"].append({
                "refnr": refnr, "pred": p.get("skills_top3"), "gold": g.get("skills_top3", "")
            })
    
    # Accuracy berechnen
    for feld in erg:
        e = erg[feld]
        e["accuracy"] = e["korrekt"] / e["total"] if e["total"] else 0
    
    erg["_meta"] = {"gemeinsame_ids": len(gem), "leere_predictions": leere_count}
    return erg


def print_accuracy_tabelle(erg: dict, label: str):
    """Druckt die Accuracy-Tabelle für ein Eval-Ergebnis."""
    print(f"\n── {label} ──")
    print(f"{'Feld':<20} {'Accuracy':>10} {'n_korrekt':>12} {'n_total':>10}")
    print("─" * 56)
    for feld in ["homeoffice", "vertragsart", "erfahrungslevel", 
                 "gehalt_min_eur", "gehalt_zeitraum", "skills_top3"]:
        e = erg[feld]
        print(f"{feld:<20} {e['accuracy']:>9.1%} {e['korrekt']:>12} {e['total']:>10}")
    
    total_k = sum(erg[f]["korrekt"] for f in erg if f != "_meta")
    total_n = sum(erg[f]["total"] for f in erg if f != "_meta")
    print("─" * 56)
    print(f"{'GESAMT':<20} {total_k/total_n:>9.1%} {total_k:>12} {total_n:>10}")
    print(f"  Leere Predictions: {erg['_meta']['leere_predictions']}")


# Test: Baseline neu auswerten mit der Funktion
baseline = eval_predictions("../daten/predictions.jsonl", gold)
print_accuracy_tabelle(baseline, "BASELINE (Phase 3)")


── BASELINE (Phase 3) ──
Feld                   Accuracy    n_korrekt    n_total
────────────────────────────────────────────────────────
homeoffice               75.0%            9         12
vertragsart              91.7%           11         12
erfahrungslevel          66.7%            8         12
gehalt_min_eur           91.7%           11         12
gehalt_zeitraum          91.7%           11         12
skills_top3               8.3%            1         12
────────────────────────────────────────────────────────
GESAMT                   70.8%           51         72
  Leere Predictions: 1


## Reflexion + Diagnose

### Hypothese-Reflexion (vs. Vorhersage oben)

| Feld | Vorhersage | Tatsächlich | Bewertung |
|---|---|---|---|
| `vertragsart` | stärkstes | **91.7 %** | ✅ wie erwartet (Top 1) |
| `homeoffice` | stark | 75.0 % | ⚠️ schwächer als erwartet (z.B. Noweda-HO übersehen) |
| `gehalt_zeitraum` | stark | **91.7 %** | ✅ Konsistenzregel funktioniert |
| `skills_top3` | mittel | **8.3 %** | ❌ **dramatisch unterschätzt** — siehe Klassifikation unten |
| `erfahrungslevel` | schwächstes | 66.7 % | ✅ wie erwartet (Bottom 2) |
| `gehalt_min_eur` | schwach | **91.7 %** | ❌ überschätzt — das Modell macht "null bei Unsicherheit", was zu hoher Accuracy führt |

**Lehre:** Mein größter Fehler war die Skills-Einschätzung. Ich hatte das Set-Match-Verfahren als großzügig gesehen, aber **`machine-learning` ≠ `machine learning`** killt allein 3 Treffer. Das ist keine inhaltliche Schwäche des Modells, sondern eine Pipeline-Schwäche.

### Zwei schwächste Felder + Iterationskandidaten für Phase 4

#### 1. `skills_top3` — Accuracy 8.3 % (1 / 12)

Die 11 Fehler verteilen sich auf vier klar trennbare Klassen:

- **Schreibweise-Varianten (3 Fälle):** `machine-learning` vs. `machine learning`, `ms-office` vs. `ms office`. Inhaltlich identisch, aber Set-Match-Pipeline wertet als Disagreement. → **Pipeline-Problem**, lässt sich durch Normalisierung beheben (z.B. Bindestriche → Leerzeichen vor Set-Vergleich).
- **Ausbildung "wird gelernt"-Skills (3 Fälle):** Bei Atruvia, msg, Wilken, DS Datentechnik extrahiert das Modell `java`, `delphi`, `c#`, obwohl diese im Text als Lerngegenstand, nicht als Anforderung stehen. Ich habe sie als Annotator weggelassen. → **Schema-Problem**, das Schema definiert nicht, ob "wird in der Ausbildung vermittelt" als Skill zählt.
- **Halluzinierte Nicht-Tool-Begriffe (1 Fall):** Atruvia mit `agile`, `frameworks`, `testen`. Methoden/Konzepte, keine konkreten Tools. → **Modell-Problem** (Prompt müsste expliziter sagen: nur konkrete Tools/Sprachen, keine Methoden).
- **Echtes Skill-Disagreement (4 Fälle):** Beide wählen 3 von 5–6 möglichen Skills, aber unterschiedliche. Beispiel Noweda: ich hatte `r|sql|machine learning`, Modell `python|r|sql`. Python ist im Anzeigentext gar nicht erwähnt → Modell halluziniert vermutlich aus Branchenwissen. → teils **Modell-Problem** (Halluzination), teils inhärente Unschärfe des Schemas (keine eindeutige Top-3-Auswahl-Regel).

**Iterations-Vermutung für Phase 4:** Skills-Pipeline normalisieren + Prompt schärfen ("Nenne nur konkrete im Text genannte Tools, keine Methoden") → Accuracy sollte deutlich steigen.

#### 2. `erfahrungslevel` — Accuracy 66.7 % (8 / 12)

Die 4 Fehler:

- **Peagle (refnr 10001-1002678126-S):** Modell `mid`, Gold `junior`. Im Text steht eindeutig *"Gerne erste Berufserfahrung nach dem Studium (maximal 2 Jahre)"*. Das ist die klarste Junior-Definition möglich. → **Modell-Problem** (übergewichtet das Wort "Berufserfahrung", ignoriert "maximal 2 Jahre").
- **KOSATEC (refnr 13644-294294-S):** Modell `mid`, Gold `nicht_genannt`. Genau der Edge Case aus Phase 2 (Annotator A vs. Annotator B). → **Schema-Problem** (siehe Phase-2-Edge-Cases).
- **Allianz (refnr 15719-44156922-55-S):** Modell `None`, Gold `nicht_genannt`. Truncation hat den Profil-Block abgeschnitten. → **Pipeline-Problem** (bereits in `02_extract.ipynb` diagnostiziert).
- **Noweda (refnr 19384-938440348-S):** Modell `nicht_genannt`, Gold `junior`. Im Text *"Erste Berufserfahrung in Data-Science-Projekten ist erwünscht, jedoch nicht zwingend erforderlich"* → klare Junior-Anzeige. → **Modell-Problem** (interpretiert "nicht zwingend" als "nicht_genannt").

**Iterations-Vermutung für Phase 4:** Im Prompt explizite Beispiele für "≤2 Jahre / erste Berufserfahrung" → junior. Zusätzlich Truncation-Strategie überdenken (Allianz fängt sich dann mit).

### Plan für Phase 4

Iteration A wird **eine** der beiden Schwächen gezielt angehen — ich entscheide dort (mit Hypothese vorab), welche Aktion erwartbar den größten Hebel hat.

## Phase 4 — Iteration A: Smart Truncation (Kopf + Schwanz)

### Hypothese vor der Iteration

**Hebel-Wahl:** Pre-Processing (eine der vier vorgegebenen Optionen aus dem Aufgabenblatt).

**Begründung aus Phase 3:** Die Allianz-Anzeige (`15719-44156922-55-S`) lieferte leere Pflichtfelder, weil 71 % des Texts (4917 von 6917 Zeichen) durch die 2000-Zeichen-Head-Truncation abgeschnitten wurden — inklusive des kompletten Profil/Anforderungs-/Skill-Blocks. Klares Pipeline-Problem.

**Erster Versuch (verworfen):** Naives Hochsetzen `MAX_TEXT_LAENGE` 2000 → 4000. Resultat: **CUDA out of memory** (V100 mit 32 GB ist bereits bei 2000 Zeichen Kontext auf der Kante; bei 4000 reicht nicht mal mehr Platz für die Inferenz-Allocation von 22 MiB). Das ist ein wertvoller Befund: Truncation hochsetzen ist auf dieser Hardware keine Option.

**Aktion (umgesetzt):** Smart Truncation bei gleichem Token-Budget. Bei Anzeigen >2000 Zeichen: erste 1200 + letzte 800 Zeichen behalten, Mitte mit Marker `[...Mitte gekürzt...]` ersetzen. Damit bleibt Speicherbedarf gleich, aber Texte wie Allianz behalten auch den Profil-/Skill-Block aus dem Ende statt nur das Boilerplate-Intro.

**Annahme dahinter:** Bei deutschen Stellenanzeigen folgt typische Struktur "Über uns" (Boilerplate) → "Aufgaben" → "Profil/Anforderungen" → "Wir bieten" (Boilerplate). Anfang + Ende ist Boilerplate, Mitte ist Inhalt. Schwanz-Truncation ist nicht ideal, aber für unsere Hand-Gold-Anzeigen ist die Erwartung: Anforderungen stehen oft erst im letzten Drittel, weil im ersten Drittel das Unternehmen vorgestellt wird.

**Erwartete Wirkung:**
- Allianz wird vollständige Predictions liefern (vorher: 3 leere Pflichtfelder)
- Andere 11 Anzeigen: keine Änderung (waren < 2000 Zeichen oder nahe dran)
- Gesamt-Accuracy steigt von 70.8 % auf ~75–77 % (+3–4 Pt absolut)
- Falls die Mitte den entscheidenden Block enthielt: keine Verbesserung oder sogar Regression bei einzelnen Feldern.

**Diagnose-Erwartung:** Bei Verbesserung → Pipeline-Problem bestätigt. Bei Stagnation → Hypothese widerlegt, dann müssen wir bei der Diagnose-Sektion ehrlich notieren, dass Smart Truncation am eigentlichen Problem vorbeigreift.

In [6]:
"""
Iteration A — neue Predictions mit MAX_TEXT_LAENGE=4000 erzeugen.

Dafür müssen wir Modell, Tokenizer und Helper-Funktionen aus 02_extract.ipynb
hier wieder laden. Da der Kernel von 03_eval.ipynb noch keine GPU/Modell-Setup hat,
geht das nicht direkt — der bessere Weg:

PRAGMATISCH: Wir bearbeiten 02_extract.ipynb, ändern MAX_TEXT_LAENGE auf 4000,
ändern den Output-Pfad zu predictions_iterA.jsonl, lassen den Loop neu laufen.
Dann hier nur evaluieren.
"""

# Wenn predictions_iterA.jsonl existiert → evaluieren
import os
iterA_pfad = "../daten/predictions_iterA.jsonl"

if not os.path.exists(iterA_pfad):
    print(f"⚠ {iterA_pfad} existiert noch nicht.")
    print("\n→ Geh zu 02_extract.ipynb:")
    print("  1. Ändere `MAX_TEXT_LAENGE = 2000` auf `MAX_TEXT_LAENGE = 4000`")
    print("  2. Ändere `ausgabe_pfad = Path(\"../daten/predictions.jsonl\")`")
    print("     auf `ausgabe_pfad = Path(\"../daten/predictions_iterA.jsonl\")`")
    print("  3. Loop-Zelle (Zelle 7) ausführen, ~60s warten")
    print("  4. Hierher zurückkehren, diese Zelle nochmal ausführen")
else:
    iterA = eval_predictions(iterA_pfad, gold)
    print_accuracy_tabelle(iterA, "ITERATION A (Truncation 4000)")


── ITERATION A (Truncation 4000) ──
Feld                   Accuracy    n_korrekt    n_total
────────────────────────────────────────────────────────
homeoffice               66.7%            8         12
vertragsart              91.7%           11         12
erfahrungslevel          75.0%            9         12
gehalt_min_eur           91.7%           11         12
gehalt_zeitraum          91.7%           11         12
skills_top3               8.3%            1         12
────────────────────────────────────────────────────────
GESAMT                   70.8%           51         72
  Leere Predictions: 1


In [7]:
"""
Iteration A — was hat sich konkret verändert ggü. Baseline?
Welche Anzeigen sind jetzt richtig (gewonnen), welche jetzt falsch (verloren)?
"""

def vergleiche_runs(run_a: dict, run_b: dict, label_a: str, label_b: str):
    """Zeigt pro Feld die ID-Verschiebungen zwischen zwei Eval-Ergebnissen."""
    print(f"\n{'='*80}")
    print(f"Verschiebungen: {label_a}  →  {label_b}")
    print(f"{'='*80}")
    
    for feld in ["homeoffice", "vertragsart", "erfahrungslevel",
                 "gehalt_min_eur", "gehalt_zeitraum", "skills_top3"]:
        fehler_a = {f["refnr"] for f in run_a[feld]["fehler"]}
        fehler_b = {f["refnr"] for f in run_b[feld]["fehler"]}
        gewonnen = fehler_a - fehler_b   # vorher falsch, jetzt richtig
        verloren = fehler_b - fehler_a   # vorher richtig, jetzt falsch
        
        if gewonnen or verloren:
            print(f"\n{feld}:")
            for refnr in sorted(gewonnen):
                print(f"  ✓ GEWONNEN  {refnr}")
            for refnr in sorted(verloren):
                # Detail: was war Pred, was war Gold im jeweiligen Run?
                fehler_detail = next(f for f in run_b[feld]["fehler"] if f["refnr"] == refnr)
                print(f"  ✗ VERLOREN  {refnr}  "
                      f"(jetzt Pred={fehler_detail['pred']!r} vs Gold={fehler_detail['gold']!r})")


vergleiche_runs(baseline, iterA, "Baseline", "Iter A")


Verschiebungen: Baseline  →  Iter A

homeoffice:
  ✓ GEWONNEN  10000-1185445644-S
  ✗ VERLOREN  10001-1003030981-S  (jetzt Pred='teilweise' vs Gold='nicht_genannt')
  ✗ VERLOREN  19384-938440348-S  (jetzt Pred='nicht_genannt' vs Gold='teilweise')

erfahrungslevel:
  ✓ GEWONNEN  10001-1002678126-S
  ✓ GEWONNEN  19384-938440348-S
  ✗ VERLOREN  13319-868490/1_607417LS-S  (jetzt Pred='egal' vs Gold='senior')

gehalt_min_eur:
  ✓ GEWONNEN  16818-100787927-S
  ✗ VERLOREN  10001-1002678126-S  (jetzt Pred=40000 vs Gold='')


### Auswertung Iteration A

**Δ Gesamt:** 0 Pt (51 → 51 korrekte Felder bei 72 total). **Δ schwächstes Feld (skills_top3):** 0 Pt — Schreibweisen-Problem unverändert. **Allianz hat weiterhin leere Predictions** (1 leeres Anzeige unverändert).

**Bewegungen unter der Oberfläche** (vergleiche-Tabelle oben):

| Feld | Gewinne | Verluste | Netto |
|---|---|---|---|
| homeoffice | +1 (Lantenhammer "Home Office möglich" jetzt im Schwanz) | −2 (Atruvia: "IT feels like home" als `teilweise` halluziniert; Noweda: "3 Tage HO/Woche" durch Mitten-Drop verloren) | **−1** |
| erfahrungslevel | +2 (Peagle "max 2 Jahre" → `junior`; Noweda "erste Erfahrung erwünscht" → `junior`) | −1 (Hays: vorher korrekt `senior`, jetzt halluziniert `egal`) | **+1** |
| gehalt_min_eur | +1 (Wilken 953€ Ausbildungsjahr jetzt erkannt) | −1 (Peagle: halluziniert 40000€ wo Gold leer ist) | **0** |

**Hypothese teilweise widerlegt:**
- Erwartung war: Allianz wird mit Smart Truncation vollständig extrahiert. Tatsächlich: Allianz hat weiterhin leere Pflichtfelder. Vermutlich liegt der Profil/Skill-Block bei Allianz **in der Mitte des Texts** (Zeichen 2000–6000), nicht am Ende — Smart Truncation greift am eigentlichen Problem vorbei.
- Erwartung war: andere 11 Anzeigen bleiben unverändert. Tatsächlich: **5 von 11** haben sich verändert (3× Gewinn, 2× Verlust), nur netto Δ=0.

**Diagnose:** Smart Truncation als Hebel **berührt** die Pipeline an mehreren Stellen gleichzeitig — sie liefert mehr Kontext am Schwanz, droppt aber wichtige Mitten-Stellen. Das Modell wird "mutiger im Raten" (mehr Halluzinationen wie 40000€, `egal`, `teilweise` aus Marketing-Slogan).

**Klassifikation:** Pipeline-Eingriff mit unspezifischem Effekt — Δ Gesamt ≈ 0 ist hier kein Zufall, sondern strukturell: die Aktion ist zu breit für die spezifischen Probleme.

**Statistische Selbstkritik:** Bei n=12 entspricht 1 Anzeige = 8.3 Pt. Alle gemessenen Δ sind ±1 Anzeige. Das ist **innerhalb des Rauschens** — die einzelnen Bewegungen sind real (siehe konkrete refnrs oben), aber bei dieser Stichprobengröße sollte man aus +1/−1 Pt keine starken Schlüsse ziehen. Was wir mit Sicherheit sagen können: **Allianz ist immer noch leer**, das ist der robuste Befund.

**Konsequenz für Iteration B:** Pre-Processing ist nicht der richtige Hebel. Iteration B sollte direkt am Modell ansetzen — **Prompt-Klarstellung** mit expliziten Beispielen für die häufigen Fehler-Klassen (Ausbildung-Skills, Junior-Erkennung, Halluzinations-Vermeidung).

## Phase 4 — Iteration B: Prompt-Klarstellung mit expliziten Anti-Halluzinations-Regeln

### Hypothese vor der Iteration

**Hebel-Wahl:** Prompt-Klarstellung (zweiter der vier Hebel aus dem Aufgabenblatt). Pre-Processing aus Iteration A war zu breit — wir gehen jetzt direkt am Modell-Verhalten ran.

**Rationale:** Iteration A hat gezeigt, dass die Bewegungen unter der Oberfläche stattfinden — Modell halluziniert mutiger (40000 €, `egal`, `teilweise` bei Marketing-Slogan). Diese Halluzinationen sind **Modell/Prompt-Problem**, nicht Pre-Processing-Problem. Lösungsansatz: dem Modell **explizit verbieten**, was es bei uns gerade halluziniert.

**Aktion:** System-Prompt um folgende harte Regeln ergänzen (jede aus einer konkreten beobachteten Fehlerklasse abgeleitet):

| Regel | Aus welchem Fehler? |
|---|---|
| "Wenn keine konkrete Zahl im Text steht, IMMER `null` für `gehalt_min_eur` — niemals raten oder schätzen" | Peagle Iter A: 40000 € halluziniert |
| "Bei Ausbildungen: skills_top3 nur Tools, die als Bewerbungsvoraussetzung gefordert sind. NICHT Tools, die 'in der Ausbildung gelernt werden'" | Atruvia/msg/Wilken: java/delphi/c# extrahiert obwohl "wird gelernt" |
| "skills_top3 nur konkrete Tools/Sprachen (Python, SQL, Spark, AWS, Java). NICHT Methoden (`agile`, `frameworks`, `testen`, `machine learning`) — diese sind Konzepte, keine Tools" | Atruvia: agile/frameworks/testen |
| "`egal` nur wenn Text explizit sagt 'Junior bis Senior willkommen' o.ä. NICHT als Standardwert" | Hays Iter A: egal halluziniert |
| "junior-Signale: 'erste Berufserfahrung', '≤2 Jahre', 'Berufseinsteiger', 'Studienabsolvent', 'Ausbildung'. Bei diesen Signalen IMMER junior, niemals mid/nicht_genannt" | Peagle Baseline + Noweda |
| "`teilweise` nur bei konkreter Hybrid-Erwähnung (Tage/Woche, 'hybrid', 'Home Office möglich'). NICHT bei Marketing-Slogans ('IT feels like home') oder 'flexibler Arbeitszeit'" | Atruvia Iter A + KOSATEC Baseline |

**Truncation:** Zurück auf einfache Head-Truncation `MAX_TEXT_LAENGE = 2000` (wie Baseline). Smart Truncation aus Iter A wird verworfen, damit der Prompt-Effekt isoliert messbar bleibt.

**Erwartete Wirkung:**
| Feld | Δ-Erwartung | Begründung |
|---|---|---|
| `skills_top3` | +25 Pt (3 Treffer) | Halluzinations-Verbot sollte agile/frameworks/testen abfangen + Bindestriche bleiben ungelöst → kein voller Sprung |
| `erfahrungslevel` | +16 Pt (2 Treffer) | Explizite Junior-Regel sollte Peagle + Noweda retten |
| `gehalt_min_eur` | 0 Pt | Wilken-953€ bleibt schwierig (Prompt allein hilft kaum); Peagle-40k war Iter-A-Artefakt |
| `homeoffice` | +8 Pt (1 Treffer) | Falsche `teilweise`-Auslöser werden seltener |
| `vertragsart`, `gehalt_zeitraum` | 0 Pt | bereits 91.7 % |

**Gesamt-Erwartung:** +4–6 Pt absolut, Gesamt-Accuracy von 70.8 % auf 76–78 %.

**Statistische Selbstkritik:** Bei n=12 ist ±8 Pt eine Anzeige. Erwartung 76% wäre 55 statt 51 korrekte Felder = +4 Anzeigen-Bewegungen. Realistischer Effekt im Rauschen, aber wir messen mehr Δ-Muster (welche Anzeigen gewonnen) als nur Δ Gesamt.

**Diagnose-Erwartung:**
- Wenn skills/erfahrungslevel signifikant steigen → Prompt-Klarstellung war richtiger Hebel, Modell-Problem bestätigt
- Wenn nur marginal → Modell-Limit erreicht, weitere Iterationen brauchen anderes Modell (3B vs 7B, in Phase 6) oder Schema-Überarbeitung
- Allianz bleibt vermutlich leer — Truncation-Problem bleibt unverändert

In [8]:
"""
Iteration B — Eval mit dem verschärften Prompt.
Erst ausführbar, wenn predictions_iterB.jsonl aus 02_extract.ipynb existiert.
"""
import os
iterB_pfad = "../daten/predictions_iterB.jsonl"

if not os.path.exists(iterB_pfad):
    print(f"⚠ {iterB_pfad} existiert noch nicht.")
    print("→ Zuerst 02_extract.ipynb mit dem neuen Prompt durchlaufen lassen.")
    print("  MAX_TEXT_LAENGE auf 1500 reduzieren wegen GPU-Memory!")
else:
    iterB = eval_predictions(iterB_pfad, gold)
    print_accuracy_tabelle(iterB, "ITERATION B (Verschärfter Prompt)")
    
    # Vergleich gegen Baseline + Iter A
    vergleiche_runs(baseline, iterB, "Baseline", "Iter B")
    vergleiche_runs(iterA, iterB, "Iter A", "Iter B")


── ITERATION B (Verschärfter Prompt) ──
Feld                   Accuracy    n_korrekt    n_total
────────────────────────────────────────────────────────
homeoffice               75.0%            9         12
vertragsart             100.0%           12         12
erfahrungslevel          75.0%            9         12
gehalt_min_eur           66.7%            8         12
gehalt_zeitraum          91.7%           11         12
skills_top3              16.7%            2         12
────────────────────────────────────────────────────────
GESAMT                   70.8%           51         72
  Leere Predictions: 0

Verschiebungen: Baseline  →  Iter B

homeoffice:
  ✓ GEWONNEN  13319-868490/1_607417LS-S
  ✗ VERLOREN  19384-938440348-S  (jetzt Pred='nicht_genannt' vs Gold='teilweise')

vertragsart:
  ✓ GEWONNEN  15719-44156922-55-S

erfahrungslevel:
  ✓ GEWONNEN  15719-44156922-55-S

gehalt_min_eur:
  ✗ VERLOREN  10001-1002026714-S  (jetzt Pred=0 vs Gold='')
  ✗ VERLOREN  10001-1003030981-S

### Auswertung Iteration B

**Δ Gesamt:** 0 Pt (51 → 51 korrekte Felder bei 72 total) — wie Iter A. **Aber:** der Mechanismus ist diesmal grundsätzlich anders.

**Was sich auf Feld-Ebene wirklich verändert hat:**

| Feld | Baseline | Iter B | Δ | Bewertung |
|---|---|---|---|---|
| `vertragsart` | 91.7 % | **100 %** | +8.3 Pt | ✅ Hypothese bestätigt — Allianz jetzt korrekt |
| `erfahrungslevel` | 66.7 % | 75 % | +8.3 Pt | ✅ Allianz jetzt korrekt (`nicht_genannt`) |
| `skills_top3` | 8.3 % | 16.7 % | +8.3 Pt | ✅ Eine Ausbildung jetzt korrekt leer (DS Datentechnik) — Ausbildungs-Regel hat gegriffen |
| `gehalt_min_eur` | 91.7 % | **66.7 %** | **−25 Pt** | ❌ Neue Fehlerklasse: Modell schreibt jetzt `0` statt `null` bei drei Ausbildungen |
| `homeoffice`, `gehalt_zeitraum` | unverändert | | 0 | — |
| **Leere Predictions** | 1 (Allianz) | **0** | — | ✅ Bonus-Effekt: schlankerer Kontext rettet Allianz |

**Der gehalt-Regression-Befund:**

Die harte Regel im Prompt *"Wenn keine konkrete Zahl im Text steht, IMMER null"* hat unerwartet zu falscher Interpretation geführt — Modell schreibt jetzt JSON-Wert `0` statt `null` bei Ausbildungen ohne Gehaltsangabe (DS Datentechnik, Atruvia, Noweda). Das ist ein klassischer **Prompt-Nebeneffekt**: das Modell hat die Regel zwar verstanden, aber falsch implementiert — vermutlich interpretiert es "null = 0 = keine Zahl" im natürlichsprachlichen Sinn statt als JSON-Schlüsselwort `null`.

**Klassifikation:** **Modell-Problem mit Prompt als Auslöser**. Eine zusätzliche Regel im Prompt ("schreibe immer das JSON-Schlüsselwort `null`, niemals die Zahl `0`") würde das vermutlich fixen — wäre dann aber Iteration C.

**Was funktioniert hat:**
- Allianz wird jetzt vollständig klassifiziert (3 zusätzliche Felder korrekt — `vertragsart`, `erfahrungslevel`, `homeoffice` jetzt mit Werten gefüllt). Dabei hilft **paradoxerweise** die Truncation-Reduktion auf 1500: weniger Kontext = schlankerer KV-Cache = Modell hat Kapazität für Output.
- Skills-Halluzination teilweise eingedämmt (Ausbildung-DS-Datentechnik jetzt korrekt leer).
- `vertragsart` jetzt bei 100 % — Schema-Konsequenz vollständig.

**Was NICHT funktioniert hat:**
- Skills-Schreibweisen-Problem (`machine-learning` vs. `machine learning`) bleibt unverändert. Das ist Pipeline-Problem, kein Prompt-Problem — hätten wir mit Iter B auch nicht erwartet zu lösen.
- Skills-Halluzination bei Festanstellungen (z.B. Hays mit `python|machine-learning|sql` statt Gold) unverändert.

**Hardware-Constraint-Diskussion:** Der OOM beim ersten Iter-B-Versuch zeigt: auf gauss-V100 32 GB ist 7B-fp32 nicht mit beliebig langen Prompts kombinierbar. Texttruncation 1500 + Prompt 2500 ≈ Maximum. Methodisch sauberer wäre ein größeres GPU-Budget — auf der vorhandenen Hardware ist das eine harte Schranke.

**Diagnose:** Prompt-Klarstellung wirkt — aber mit Nebeneffekten. Eine 3. Iteration könnte die `null`-vs-`0`-Sache lösen, dann hätten wir vermutlich ~78 % Gesamt-Accuracy. Aber: das ist Phase-5/6-Territory. Für Phase 4 ist die Lehre wichtiger: **Hebel zeigen sich erst in der Praxis als zu breit/zu spitz**.

## Phase 4 — Synthese

### Iterations-Tabelle

| Iteration | Hypothese | Aktion | Δ Gesamt | Δ schwächstes Feld (skills) | Diagnose |
|---|---|---|---|---|---|
| Baseline | — | Head-Truncation 2000, Standard-Prompt | 70.8 % | 8.3 % | — |
| A | Allianz-Problem ist Truncation → Smart Truncation hilft | Anfang 1200 + Ende 800, Mitte droppen | 0 Pt (70.8 %) | 0 Pt (8.3 %) | Pipeline-Eingriff mit **unspezifischem** Effekt: 3 Anzeigen gewonnen, 3 verloren — netto 0. Allianz weiterhin leer → Hypothese widerlegt: Profil-Block liegt mittig, nicht am Ende. |
| B | Halluzinationen sind Modell/Prompt-Problem → harte Regeln helfen | Prompt um Anti-Halluzinations-Regeln erweitert, Texttruncation 1500 wegen GPU-Memory | 0 Pt (70.8 %) | +8.3 Pt (16.7 %) | Mehrere Felder gewinnen (`vertragsart` 100 %, `erfahrungslevel` +1, `skills` +1, Allianz nicht mehr leer), aber `gehalt_min_eur` kollabiert −25 Pt durch **neuen Halluzinations-Typ** (Modell schreibt `0` statt `null`). Nettoeffekt zufällig 0 — aber Mechanismus diesmal nachvollziehbar. |

### Synthese-Antworten

**1. Welcher der drei Fehler-Typen (Schema / Modell / Pipeline) war in unserem Setup die häufigste Ursache?**

In unserem Baseline-Lauf waren alle drei Typen vertreten, aber **Modell-Probleme dominierten**. Konkrete Verteilung der Phase-3-Fehler:

- **Modell-Probleme (am häufigsten):** Modell missachtet klare Textsignale wie "max. 2 Jahre" oder "Ausbildungsjahr: 953 €"; halluziniert Skills wie "agile" oder "frameworks"; "wird in der Ausbildung gelernt"-Tools werden extrahiert.
- **Pipeline-Probleme (1 Fall):** Allianz-Truncation (3 leere Felder gleichzeitig).
- **Schema-Probleme (2 Fälle):** KOSATEC `nicht_genannt` vs. `mid` (Edge Case aus Phase 2); Schema definiert nicht, ob "wird gelernt"-Skills bei Ausbildungen zählen.

Iteration B hat gezeigt: **Prompt-Eingriffe können Modell-Probleme adressieren, aber jeder neue Prompt-Hebel kann neue Halluzinationen erzeugen.** Modell-Probleme sind keine einmaligen Fehler, sondern Verhaltensmuster, die sich durch jede Prompt-Änderung in andere Verhaltensmuster transformieren.

**2. Was wäre nicht durch Iteration lösbar — wo ist Modell oder Schema die harte Grenze?**

- **Schema-Grenze:** Die KOSATEC-Frage `nicht_genannt` vs. `mid` (Edge Case 1 aus Phase 2) ist nicht durch Iteration auflösbar — das Schema definiert "Erfahrung erwähnt aber ohne Jahresangabe" nicht eindeutig. Da hilft nur Schema-Update.
- **Modell-Grenze:** Bei nur 7 Mrd. Parametern kommt das Modell mit subtilen Junior/Mid/Senior-Übergängen an seine Grenze, besonders wenn keine Jahresangabe da ist. Ein größeres Modell (Frontier in Phase 5, 14B/Claude) würde hier vermutlich besser sein — aber das ändert nichts am inhärenten Schema-Problem.
- **Hardware-Grenze:** Auf gauss-V100-32GB ist die Kombination "großer Prompt + langer Kontext" nicht parallel möglich. Wir mussten in Iter B die Truncation reduzieren, um den Prompt erweitern zu können. Das ist eine echte Schranke, die durch besseres Modell-Hosting (mehr Speicher, fp8/int4) gelöst werden müsste.

**3. Würden wir im Berufsalltag alle Probleme fixen, oder gibt es Fehler, die "gut genug" sind?**

Im Berufsalltag würden wir **nach Use Case priorisieren**:

- **Kritisch zu fixen:** `gehalt_min_eur` und `gehalt_zeitraum`. Diese Felder werden für Filtersuchen genutzt — falsche Werte hier kosten Bewerber und Arbeitgeber direkt. Aktuelle Genauigkeit 66-91 % ist nicht gut genug für Produktiv-Einsatz; eine `0`-statt-`null`-Verwechslung wäre fatal (z.B. "Stellen unter 500 €/Monat" würde fälschlich alle ohne Gehaltsangabe matchen).
- **"Gut genug" toleriert:** `skills_top3` bei 16-50 %. Skills werden meist als Anzeige-Suggestion oder Tag-Cloud verwendet — wenn 2 von 3 Skills passen, ist das oft akzeptabel. Schreibweisen-Varianten ("machine-learning" vs. "machine learning") sind kosmetisch.
- **Schwellwert für unsere Pipeline:** Wir würden in Produktion erst ab ≥95 % pro Feld einsetzen, mit Confidence-Score und menschlichem Review für niedrige Confidence. Aktuelle 70 % Gesamt-Accuracy ist Prototype-Qualität, kein Produkt.

### Statistische Selbstkritik

Bei n=12 entspricht eine einzelne falsch klassifizierte Anzeige ≈ 8.3 Pt.

- Iter A: Δ Gesamt = 0, aber Δ-Muster = 6 Anzeigen-Bewegungen → Aktion **wirkt**, aber nicht in die gewünschte Richtung.
- Iter B: Δ Gesamt = 0, aber Δ-Muster = 7 Anzeigen-Bewegungen mit klar interpretierbarem Mechanismus (Allianz gerettet, Gehalt-Halluzination neu).
- **Beide Iterationen scheinen oberflächlich wirkungslos (Δ Gesamt = 0), sind aber bei der Δ-Muster-Analyse höchst lehrreich.** Δ Gesamt allein ist bei n=12 keine ausreichende Metrik.

Für robustere Aussagen bräuchte es n ≥ 30 Gold-annotierte Anzeigen — dann wäre eine einzelne Anzeige nur noch ≈ 3.3 Pt und Effekte über 10 Pt wären klar signifikant. Ein zukünftiger Workflow sollte mit größerer Gold-Stichprobe arbeiten.

### Wäre eine 3. Iteration sinnvoll?

Inhaltlich ja: das `0`-vs-`null`-Problem aus Iter B ist eine 1-Zeilen-Prompt-Ergänzung ("schreibe immer das JSON-Schlüsselwort `null`, niemals die Zahl `0`") und würde drei Anzeigen retten → erwartete Gesamt-Accuracy 78 %. Aber das ist Phase 5/6-Material und sprengt den 4-h-Zeitrahmen von Phase 4. Wir notieren es als Erkenntnis und gehen mit Iteration A + B in die Frontier-Vergleich-Phase.

## Phase 6 — Vollständiger 7B-Run + 3B-Halluzinations-Klassen

Per-Field-Accuracy auf den 12 Hand-Gold-Anzeigen + Schema-Konformitäts-Check auf den restlichen Anzeigen ohne Gold. 3B-vs-7B-vs-Gold per `refnr` joinen, drei eigenständige Halluzinations-Klassen mit konkreten Beispielen identifizieren.